<a href="https://colab.research.google.com/github/jen257/CS4287_CamVid_FCN/blob/main/CS4287_Assign2_23365684_23033614_23110465.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS4287 Assignment 1: CNNs
## Team Members:
1. HuiLin Chong (ID: 23365684)
2. JiaLiang Lyu (ID: 23033614)
3. JiaDong Chen (ID: 23110465)


## Execution Status:


## Third-Party Sources:
https://github.com/jen257/CS4287_CamVid_FCN

https://github.com/alexgkendall/SegNet-Tutorial.git

In [2]:
!git clone https://github.com/alexgkendall/SegNet-Tutorial.git

Cloning into 'SegNet-Tutorial'...
remote: Enumerating objects: 2785, done.
remote: Total 2785 (delta 0), reused 0 (delta 0), pack-reused 2785 (from 1)
Receiving objects: 100% (2785/2785), 340.84 MiB | 34.42 MiB/s, done.
Resolving deltas: 100% (81/81), done.


# Task

###Task 1: Develop Dummy Data Placeholder DataLoader

In [3]:
import torch
from torch.utils.data import Dataset, DataLoader

# Define global constants directly instead of importing from config.py
BATCH_SIZE = 8
NUM_CLASSES = 12
IMG_HEIGHT = 360
IMG_WIDTH = 480


# Define a placeholder Dataset to test the pipeline before real data is loaded
class FakeCamVidDataset(Dataset):
    def __init__(self, size=32):
        # Total number of dummy samples
        self.size = size

    def __len__(self):
        # Return the total dataset size
        return self.size

    def __getitem__(self, idx):
        # Generate random RGB image tensor with shape (3, Height, Width) in range [0, 1)
        image = torch.rand(3, IMG_HEIGHT, IMG_WIDTH, dtype=torch.float32)

        # Generate random mask with integer class labels [0, NUM_CLASSES - 1]
        # Shape: (Height, Width)
        mask = torch.randint(0, NUM_CLASSES, (IMG_HEIGHT, IMG_WIDTH), dtype=torch.long)

        return image, mask


# Instantiate dataset and dataloader for stage 1 interface testing
fake_dataset = FakeCamVidDataset(size=32)
fake_loader = DataLoader(fake_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Quick sanity check: verify output shapes of one batch
if __name__ == "__main__":
    for images, masks in fake_loader:
        print("Batch Image Shape:", images.shape)  # Expected: [8, 3, 360, 480]
        print("Batch Mask Shape :", masks.shape)   # Expected: [8, 360, 480]
        break

Batch Image Shape: torch.Size([8, 3, 360, 480])
Batch Mask Shape : torch.Size([8, 360, 480])


###Task 2: Implement FCN-32s without Skip Connections and Test Forward Pass with Dummy Data

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Configuration matching config.py requirements
BATCH_SIZE = 8
NUM_CLASSES = 12
IMG_HEIGHT = 360
IMG_WIDTH = 480


class FCN32s(nn.Module):

  def __init__(self, num_classes=NUM_CLASSES):
    super(FCN32s, self).__init__()

    # Layer-by-layer VGG-style feature extractor
    self.features = nn.Sequential(
        # Block 1 (Downsampling: 1/2)
        nn.Conv2d(3, 64, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(64, 64, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2),
        # Block 2 (Downsampling: 1/4)
        nn.Conv2d(64, 128, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(128, 128, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2),
        # Block 3 (Downsampling: 1/8)
        nn.Conv2d(128, 256, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(256, 256, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2),
        # Block 4 (Downsampling: 1/16)
        nn.Conv2d(256, 512, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(512, 512, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2),
        # Block 5 (Downsampling: 1/32)
        nn.Conv2d(512, 512, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(512, 512, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2),
    )

    # Fully convolutional classifier layers (fc6, fc7 replaced with 1x1 conv)
    self.classifier = nn.Sequential(
        nn.Conv2d(512, 512, kernel_size=1),
        nn.ReLU(inplace=True),
        nn.Dropout2d(p=0.5),
        nn.Conv2d(512, num_classes, kernel_size=1),
    )

    # 32x upsampling using transposed convolution
    self.upsample = nn.ConvTranspose2d(
        in_channels=num_classes,
        out_channels=num_classes,
        kernel_size=32,
        stride=32,
        bias=False,
    )

  def forward(self, x):
    # Extract features (downsampled by 32)
    feat = self.features(x)

    # Predict class scores per pixel in latent space
    score = self.classifier(feat)

    # Upsample back
    out = self.upsample(score)

    # Resize/crop to strictly match input dimensions (360x480) dynamically
    out = F.interpolate(out, size=(x.shape[2], x.shape[3]), mode="bilinear", align_corners=False)

    # Return logits without softmax
    return out

# Task 2: Dummy Data Forward Pass Test
if __name__ == "__main__":
  # Initialize model
  model = FCN32s(num_classes=NUM_CLASSES)
  model.eval()

  # Create dummy input: (BATCH_SIZE, 3, IMG_HEIGHT, IMG_WIDTH)
  dummy_images = torch.randn(
      BATCH_SIZE, 3, IMG_HEIGHT, IMG_WIDTH, dtype=torch.float32
  )

  # Forward pass
  with torch.no_grad():
    output_logits = model(dummy_images)

  # Verify output tensor shape: (BATCH_SIZE, NUM_CLASSES, IMG_HEIGHT, IMG_WIDTH)
  print(f"Input shape:  {dummy_images.shape}")
  print(f"Output shape: {output_logits.shape}")

  # Check dimensions match project specification
  assert output_logits.shape == (
      BATCH_SIZE,
      NUM_CLASSES,
      IMG_HEIGHT,
      IMG_WIDTH,
  ), "Dimension mismatch!"
  print("Test passed successfully!")

Input shape:  torch.Size([8, 3, 360, 480])
Output shape: torch.Size([8, 12, 360, 480])
Test passed successfully!


###Task 3: Implement mIoU Calculation Function and Basic Training Loop, Test Backpropagation with Dummy Data

In [5]:
import torch
import torch.nn as nn

# 1. Setup Dummy Data
BATCH_SIZE = 8
NUM_CLASSES = 12
IMG_HEIGHT = 360
IMG_WIDTH = 480

print("--- Step 1: Preparing Tensors ---")
# FCN output: (Batch, Classes, Height, Width)
fake_preds = torch.randn(BATCH_SIZE, NUM_CLASSES, IMG_HEIGHT, IMG_WIDTH, requires_grad=True)
print(f"Prediction shape: {fake_preds.shape}")

# Labels: 0 to 11 (Batch, Height, Width)
fake_labels = torch.randint(0, NUM_CLASSES, (BATCH_SIZE, IMG_HEIGHT, IMG_WIDTH))
print(f"Label shape: {fake_labels.shape}")

# 2. Test CrossEntropyLoss & Backprop
print("\n--- Step 2: Loss & Backprop ---")
loss_function = nn.CrossEntropyLoss()

# Calculate loss
loss = loss_function(fake_preds, fake_labels)
print(f"Loss: {loss.item():.4f}")

# Task 4 requirement: Test backpropagation
loss.backward()
print("Success: loss.backward() executed without errors!")


# 3. Simple mIoU Calculation
print("\n--- Step 3: Calculate mIoU ---")
# Get predicted class IDs (Batch, Height, Width)
preds_id = torch.argmax(fake_preds, dim=1)

iou_list = []

for class_id in range(NUM_CLASSES):
    pred_c = (preds_id == class_id)
    label_c = (fake_labels == class_id)

    # Intersection and Union
    intersection = (pred_c & label_c).sum().item()
    union = (pred_c | label_c).sum().item()

    # Avoid division by zero
    if union > 0:
        iou_list.append(intersection / union)

# Mean IoU
miou_score = sum(iou_list) / len(iou_list)
print(f"mIoU score: {miou_score:.4f}")

--- Step 1: Preparing Tensors ---
Prediction shape: torch.Size([8, 12, 360, 480])
Label shape: torch.Size([8, 360, 480])

--- Step 2: Loss & Backprop ---
Loss: 2.9210
Success: loss.backward() executed without errors!

--- Step 3: Calculate mIoU ---
mIoU score: 0.0437


###Task 4: Map RGB Colors to Class IDs, Load Real CamVid Images, and Apply Synchronized Data Augmentation

In [6]:
import os
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Configuration parameters (defined locally instead of importing from non-existent config.py)
BATCH_SIZE = 8
NUM_CLASSES = 12
IMG_HEIGHT = 360
IMG_WIDTH = 480

# 1. Official 12-class CamVid RGB color map
# Each RGB tuple maps to a class index (0 to 11)
CAMVID_COLOR_MAP = {
    (64, 128, 64): 0,    # Animal
    (192, 0, 128): 1,    # Archway
    (0, 128, 192): 2,    # Bicyclist
    (0, 128, 64): 3,     # Bridge
    (128, 0, 0): 4,      # Building
    (64, 0, 128): 5,     # Car
    (64, 64, 0): 6,      # Cart/Luggage
    (0, 0, 192): 7,      # Child
    (192, 128, 128): 8,  # Column/Pole
    (128, 64, 128): 9,   # Fence
    (0, 0, 64): 10,      # Lane/Marking
    (128, 128, 128): 11  # Misc / Void (or other class according to your split)
}

def rgb_to_mask(rgb_mask):
    """
    Convert a 3-channel RGB mask image to a 2D integer class label mask (H, W).
    """
    h, w, _ = rgb_mask.shape
    mask_2d = np.zeros((h, w), dtype=np.int64)

    # Match each RGB pixel to its class ID
    for rgb_color, class_id in CAMVID_COLOR_MAP.items():
        # Find pixels matching the RGB tuple
        matches = np.all(rgb_mask == rgb_color, axis=-1)
        mask_2d[matches] = class_id

    return mask_2d


# 2. Synchronized augmentation pipeline using albumentations
def get_transforms():
    """
    Create a transformation pipeline for both image and mask.
    """
    return A.Compose([
        A.Resize(height=IMG_HEIGHT, width=IMG_WIDTH),
        A.HorizontalFlip(p=0.5), # 50% chance to flip horizontally
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2() # Convert image to PyTorch tensor (C, H, W)
    ])


# 3. Real CamVid PyTorch Dataset
class CamVidDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.transform = transform
        # Get sorted list of file names
        self.file_names = sorted(os.listdir(images_dir))

    def __len__(self):
        return len(self.file_names)

    def __getitem__(self, idx):
        filename = self.file_names[idx]
        img_path = os.path.join(self.images_dir, filename)
        # Note: adjust mask naming if masks have a suffix like '_L.png'
        mask_path = os.path.join(self.masks_dir, filename)

        # Read images using OpenCV (convert BGR to RGB)
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask_rgb = cv2.imread(mask_path)
        mask_rgb = cv2.cvtColor(mask_rgb, cv2.COLOR_BGR2RGB)

        # Convert RGB mask to 2D class label mask
        mask_2d = rgb_to_mask(mask_rgb)

        # Apply synchronized transformations
        if self.transform:
            augmented = self.transform(image=image, mask=mask_2d)
            image = augmented["image"]
            mask_2d = augmented["mask"]

        # Ensure mask is a torch.long tensor
        mask_tensor = torch.as_tensor(mask_2d, dtype=torch.long)

        return image, mask_tensor

###Task 5: Extract Pool3/Pool4 Feature Maps and Implement FCN-8s via Addition Fusion

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Define global constants directly instead of importing from config.py
BATCH_SIZE = 8
NUM_CLASSES = 12
IMG_HEIGHT = 360
IMG_WIDTH = 480


class FCN8s(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super(FCN8s, self).__init__()
        # 1. Feature Extractor Backbone (VGG-style)
        # Block 1 (Downsampling: 1/2)
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Block 2 (Downsampling: 1/4)
        self.block2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Block 3 (Downsampling: 1/8) -> Extract Pool3
        self.block3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Block 4 (Downsampling: 1/16) -> Extract Pool4
        self.block4 = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Block 5 (Downsampling: 1/32) -> Deepest layer
        self.block5 = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # 2. Fully Convolutional Classifier
        self.classifier = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=0.5),
            nn.Conv2d(512, num_classes, kernel_size=1)
        )

        # 3. 1x1 Convolutions for Skip Connections
        # Project Pool4 and Pool3 feature channels directly to num_classes
        self.score_pool4 = nn.Conv2d(512, num_classes, kernel_size=1)
        self.score_pool3 = nn.Conv2d(256, num_classes, kernel_size=1)

        # 4. Transposed Convolutions for Upsampling
        # 2x upsampling for classifier score to match Pool4 size
        self.upsample2x = nn.ConvTranspose2d(
            num_classes, num_classes, kernel_size=4, stride=2, padding=1, bias=False
        )
        # 2x upsampling for fused (classifier + Pool4) score to match Pool3 size
        self.upsample_pool4 = nn.ConvTranspose2d(
            num_classes, num_classes, kernel_size=4, stride=2, padding=1, bias=False
        )
        # 8x upsampling for final fused score back to original image size
        self.upsample8x = nn.ConvTranspose2d(
            num_classes, num_classes, kernel_size=16, stride=8, padding=4, bias=False
        )

    def forward(self, x):
        input_h, input_w = x.shape[2], x.shape[3]

        # Backbone forward passes
        p1 = self.block1(x)
        p2 = self.block2(p1)
        p3 = self.block3(p2)  # 1/8 scale
        p4 = self.block4(p3)  # 1/16 scale
        p5 = self.block5(p4)  # 1/32 scale

        # Step 1: Deepest layer classification score
        score_p5 = self.classifier(p5)

        # Step 2: First Skip Connection (Fuse with Pool4)
        up_score_p5 = self.upsample2x(score_p5)
        score_p4 = self.score_pool4(p4)

        # Crop or interpolate to match spatial dimensions if there's any rounding difference
        if up_score_p5.shape[2:] != score_p4.shape[2:]:
            up_score_p5 = F.interpolate(up_score_p5, size=score_p4.shape[2:], mode='bilinear', align_corners=False)

        fuse_pool4 = up_score_p5 + score_p4  # Element-wise addition

        # Step 3: Second Skip Connection (Fuse with Pool3)
        up_fuse_pool4 = self.upsample_pool4(fuse_pool4)
        score_p3 = self.score_pool3(p3)

        if up_fuse_pool4.shape[2:] != score_p3.shape[2:]:
            up_fuse_pool4 = F.interpolate(up_fuse_pool4, size=score_p3.shape[2:], mode='bilinear', align_corners=False)

        fuse_pool3 = up_fuse_pool4 + score_p3  # Element-wise addition

        # Step 4: Final 8x upsampling to original image resolution
        out = self.upsample8x(fuse_pool3)

        # Strictly align to input resolution (360, 480)
        if out.shape[2:] != (input_h, input_w):
            out = F.interpolate(out, size=(input_h, input_w), mode='bilinear', align_corners=False)

        # Return unnormalized logits (no Softmax)
        return out


# Forward Pass Test
if __name__ == "__main__":
    model = FCN8s(num_classes=NUM_CLASSES)
    model.eval()

    # Create dummy batch using config dimensions
    dummy_input = torch.randn(BATCH_SIZE, 3, IMG_HEIGHT, IMG_WIDTH, dtype=torch.float32)

    with torch.no_grad():
        output = model(dummy_input)

    print(f"Input shape:  {dummy_input.shape}")
    print(f"Output shape: {output.shape}")

    expected_shape = (BATCH_SIZE, NUM_CLASSES, IMG_HEIGHT, IMG_WIDTH)
    assert output.shape == expected_shape, f"Dimension mismatch! Expected {expected_shape}, got {output.shape}"
    print("FCN-8s forward test passed successfully!")

Input shape:  torch.Size([8, 3, 360, 480])
Output shape: torch.Size([8, 12, 360, 480])
FCN-8s forward test passed successfully!


###Task 6: Implement K-Fold Splitting, Weighted Cross-Entropy/Dice Loss, Model Saving, and Plot Loss Curves

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt
import numpy as np
import copy
import os

def train_kfold(dataset, model_class, num_epochs=30, k_folds=5, batch_size=8):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    kfold = KFold(n_splits=k_folds, shuffle=True, random_state=42)

    # Class weights to handle CamVid's severe class imbalance
    weights = torch.tensor([1.0, 1.5, 2.0, 1.2, 3.0, 1.0, 1.0, 1.5, 2.5, 1.0, 1.0, 2.0], dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)

    os.makedirs('/content/drive/MyDrive/CS4287_Weights', exist_ok=True)
    fold_metrics = {}

    for fold, (train_idx, val_idx) in enumerate(kfold.split(dataset)):
        print(f"\n--- Starting Fold {fold + 1}/{k_folds} ---")

        train_loader = DataLoader(Subset(dataset, train_idx), batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(Subset(dataset, val_idx), batch_size=batch_size, shuffle=False)

        model = model_class().to(device)
        optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

        best_val_loss = float('inf')
        history = {'train_loss': [], 'val_loss': []}

        for epoch in range(num_epochs):
            model.train()
            train_loss = 0.0
            for images, masks in train_loader:
                images, masks = images.to(device), masks.to(device)

                optimizer.zero_grad()
                loss = criterion(model(images), masks)
                loss.backward()
                optimizer.step()
                train_loss += loss.item() * images.size(0)

            epoch_train_loss = train_loss / len(train_idx)
            history['train_loss'].append(epoch_train_loss)

            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for images, masks in val_loader:
                    images, masks = images.to(device), masks.to(device)
                    loss = criterion(model(images), masks)
                    val_loss += loss.item() * images.size(0)

            epoch_val_loss = val_loss / len(val_idx)
            history['val_loss'].append(epoch_val_loss)

            print(f"Epoch [{epoch+1}/{num_epochs}] Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")

            if epoch_val_loss < best_val_loss:
                best_val_loss = epoch_val_loss
                save_path = f'/content/drive/MyDrive/CS4287_Weights/best_model_fold_{fold+1}.pth'
                torch.save(model.state_dict(), save_path)
                print(f"-> Model improved. Weights saved to {save_path}")

        fold_metrics[fold+1] = best_val_loss

        plt.figure(figsize=(8, 4))
        plt.plot(history['train_loss'], label='Train Loss')
        plt.plot(history['val_loss'], label='Val Loss')
        plt.title(f'Fold {fold+1} Loss History')
        plt.xlabel('Epoch')
        plt.ylabel('Weighted Cross Entropy')
        plt.legend()
        plt.grid(True)
        plt.savefig(f'/content/drive/MyDrive/CS4287_Weights/loss_curve_fold_{fold+1}.png')
        plt.close()

    print(f"\nK-Fold Cross Validation Completed. Best Val Losses: {fold_metrics}")

In [9]:
import os
import gc
import torch
import cv2
import numpy as np
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2

# 1. Define paths and configuration constants
IMAGES_DIR = '/content/SegNet-Tutorial/CamVid/train'
MASKS_DIR = '/content/SegNet-Tutorial/CamVid/trainannot'
NUM_CLASSES = 12
IMG_HEIGHT = 360
IMG_WIDTH = 480

# Official 12-class CamVid RGB color map
CAMVID_COLOR_MAP = {
    (64, 128, 64): 0,    # Animal
    (192, 0, 128): 1,    # Archway
    (0, 128, 192): 2,    # Bicyclist
    (0, 128, 64): 3,     # Bridge
    (128, 0, 0): 4,      # Building
    (64, 0, 128): 5,     # Car
    (64, 64, 0): 6,      # Cart/Luggage
    (0, 0, 192): 7,      # Child
    (192, 128, 128): 8,  # Column/Pole
    (128, 64, 128): 9,   # Fence
    (0, 0, 64): 10,      # Lane/Marking
    (128, 128, 128): 11  # Misc / Void
}

def rgb_to_mask(rgb_mask):
    h, w, _ = rgb_mask.shape
    mask_2d = np.zeros((h, w), dtype=np.int64)
    for rgb_color, class_id in CAMVID_COLOR_MAP.items():
        matches = np.all(rgb_mask == rgb_color, axis=-1)
        mask_2d[matches] = class_id
    return mask_2d

def get_transforms():
    return A.Compose([
        A.Resize(height=IMG_HEIGHT, width=IMG_WIDTH),
        A.HorizontalFlip(p=0.5),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])

class CamVidDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.transform = transform
        self.file_names = sorted(os.listdir(images_dir))

    def __len__(self):
        return len(self.file_names)

    def __getitem__(self, idx):
        filename = self.file_names[idx]
        img_path = os.path.join(self.images_dir, filename)
        mask_path = os.path.join(self.masks_dir, filename)

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask_rgb = cv2.imread(mask_path)
        mask_rgb = cv2.cvtColor(mask_rgb, cv2.COLOR_BGR2RGB)

        mask_2d = rgb_to_mask(mask_rgb)

        if self.transform:
            augmented = self.transform(image=image, mask=mask_2d)
            image = augmented["image"]
            mask_2d = augmented["mask"]

        mask_tensor = torch.as_tensor(mask_2d, dtype=torch.long)
        return image, mask_tensor

if not os.path.exists(IMAGES_DIR) or not os.path.exists(MASKS_DIR):
    print("Error: Dataset paths not found. Please ensure the git clone step was executed.")
else:
    print("Dataset directories found. Initializing pipeline...")

    # 2. Setup transforms and dataset
    transforms = get_transforms()
    real_dataset = CamVidDataset(images_dir=IMAGES_DIR, masks_dir=MASKS_DIR, transform=transforms)

    # 3. Clear memory before training
    torch.cuda.empty_cache()
    gc.collect()

    print("Starting 5-Fold Cross Validation on CamVid dataset...")

    # 4. Execute the training loop (Member C) with FCN-8s (Member B)
    train_kfold(
        dataset=real_dataset,
        model_class=FCN8s,
        num_epochs=30,
        k_folds=5,
        batch_size=8
    )

Dataset directories found. Initializing pipeline...
Starting 5-Fold Cross Validation on CamVid dataset...

--- Starting Fold 1/5 ---
Epoch [1/30] Train Loss: 1.6133 | Val Loss: 0.0439
-> Model improved. Weights saved to /content/drive/MyDrive/CS4287_Weights/best_model_fold_1.pth
Epoch [2/30] Train Loss: 0.0081 | Val Loss: 0.0003
-> Model improved. Weights saved to /content/drive/MyDrive/CS4287_Weights/best_model_fold_1.pth
Epoch [3/30] Train Loss: 0.0002 | Val Loss: 0.0001
-> Model improved. Weights saved to /content/drive/MyDrive/CS4287_Weights/best_model_fold_1.pth
Epoch [4/30] Train Loss: 0.0001 | Val Loss: 0.0000
-> Model improved. Weights saved to /content/drive/MyDrive/CS4287_Weights/best_model_fold_1.pth
Epoch [5/30] Train Loss: 0.0000 | Val Loss: 0.0000
-> Model improved. Weights saved to /content/drive/MyDrive/CS4287_Weights/best_model_fold_1.pth
Epoch [6/30] Train Loss: 0.0000 | Val Loss: 0.0000
-> Model improved. Weights saved to /content/drive/MyDrive/CS4287_Weights/best_mo

###Task 7: Conduct Data Augmentation Ablation Study to Verify Overfitting Suppression

In [1]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader

# Removed imports from external 'config' and 'dataset' files
# since they are defined locally in the notebook.

# 1. Baseline Transformation: minimal processing (Resize + Normalize)
def get_baseline_transforms():
    return A.Compose([
        A.Resize(height=IMG_HEIGHT, width=IMG_WIDTH),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])


# 2. Strong Augmentation: geometric + color perturbations to fight overfitting
def get_strong_transforms():
    return A.Compose([
        A.Resize(height=IMG_HEIGHT, width=IMG_WIDTH),
        A.HorizontalFlip(p=0.5),                                     # Spatial flip
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, p=0.5), # Color jitter
        A.GaussNoise(p=0.5),                                         # Gaussian noise
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])


# 3. Create the two DataLoaders for Member C's ablation experiment
def get_ablation_dataloaders(train_img_dir, train_mask_dir):
    # Baseline loader
    baseline_dataset = CamVidDataset(
        images_dir=train_img_dir,
        masks_dir=train_mask_dir,
        transform=get_baseline_transforms()
    )
    baseline_loader = DataLoader(
        baseline_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    # Strong augmentation loader
    strong_dataset = CamVidDataset(
        images_dir=train_img_dir,
        masks_dir=train_mask_dir,
        transform=get_strong_transforms()
    )
    strong_loader = DataLoader(
        strong_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    return baseline_loader, strong_loader

###Task 8: Conduct Segmentation Performance Comparison: FCN-32s vs. FCN-8s

In [5]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# Color map from class ID to RGB for visualization (matching CamVid classes)
CAMVID_ID_TO_COLOR = {
    0: (64, 128, 64),     # Animal
    1: (192, 0, 128),     # Archway
    2: (0, 128, 192),     # Bicyclist
    3: (0, 128, 64),      # Bridge
    4: (128, 0, 0),       # Building
    5: (64, 0, 128),      # Car
    6: (64, 64, 0),       # Cart/Luggage
    7: (0, 0, 192),       # Child
    8: (192, 128, 128),   # Column/Pole
    9: (128, 64, 128),    # Fence
    10: (0, 0, 64),       # Lane/Marking
    11: (128, 128, 128)   # Misc / Void
}

def mask_to_rgb(mask_2d):
    # Convert a 2D class label mask (H, W) into a 3D RGB image (H, W, 3) for plotting.

    h, w = mask_2d.shape
    rgb_image = np.zeros((h, w, 3), dtype=np.uint8)
    for class_id, color in CAMVID_ID_TO_COLOR.items():
        rgb_image[mask_2d == class_id] = color
    return rgb_image


def compare_fcn_models(model_32s, model_8s, val_loader, device='cuda' if torch.cuda.is_available() else 'cpu', num_samples=1):
    # Task 8: Comparative visualization of FCN-32s and FCN-8s predictions on validation images.
    # Demonstrates how skip connections in FCN-8s recover fine details (e.g. poles, pedestrians).
    model_32s.to(device).eval()
    model_8s.to(device).eval()

    # Get one batch from validation loader
    images, masks = next(iter(val_loader))
    images = images.to(device)

    with torch.no_grad():
        # Predict logits and take argmax across channel dimension
        logits_32s = model_32s(images)
        preds_32s = torch.argmax(logits_32s, dim=1).cpu().numpy()

        logits_8s = model_8s(images)
        preds_8s = torch.argmax(logits_8s, dim=1).cpu().numpy()

    images_np = images.cpu().permute(0, 2, 3, 1).numpy()
    masks_np = masks.cpu().numpy()

    # Denormalize image for correct display (standard ImageNet mean and std)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    for i in range(min(num_samples, len(images))):
        img_vis = np.clip((images_np[i] * std + mean) * 255, 0, 255).astype(np.uint8)
        gt_vis = mask_to_rgb(masks_np[i])
        pred_32s_vis = mask_to_rgb(preds_32s[i])
        pred_8s_vis = mask_to_rgb(preds_8s[i])

        # Plot side-by-side comparison
        plt.figure(figsize=(18, 5))

        plt.subplot(1, 4, 1)
        plt.title("Input Image")
        plt.imshow(img_vis)
        plt.axis('off')

        plt.subplot(1, 4, 2)
        plt.title("Ground Truth")
        plt.imshow(gt_vis)
        plt.axis('off')

        plt.subplot(1, 4, 3)
        plt.title("FCN-32s Prediction")
        plt.imshow(pred_32s_vis)
        plt.axis('off')

        plt.subplot(1, 4, 4)
        plt.title("FCN-8s Prediction (With Skips)")
        plt.imshow(pred_8s_vis)
        plt.axis('off')

        plt.show()

In [6]:
!rm -rf /content/SegNet-Tutorial
!git clone https://github.com/alexgkendall/SegNet-Tutorial.git

Cloning into 'SegNet-Tutorial'...
remote: Enumerating objects: 2785, done.
remote: Total 2785 (delta 0), reused 0 (delta 0), pack-reused 2785 (from 1)
Receiving objects: 100% (2785/2785), 340.84 MiB | 25.95 MiB/s, done.
Resolving deltas: 100% (81/81), done.


###Task 9: Execute Full 5-Fold Cross-Validation on Optimal Configuration and Summarize Final Metrics

In [10]:
import os
import cv2
import gc
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2

# 1. Define dataset paths and configuration constants
IMAGES_DIR = '/content/SegNet-Tutorial/CamVid/train'
MASKS_DIR = '/content/SegNet-Tutorial/CamVid/trainannot'
NUM_CLASSES = 12
IMG_HEIGHT = 360
IMG_WIDTH = 480

# Official 12-class CamVid RGB color map
CAMVID_COLOR_MAP = {
    (64, 128, 64): 0,    # Animal
    (192, 0, 128): 1,    # Archway
    (0, 128, 192): 2,    # Bicyclist
    (0, 128, 64): 3,     # Bridge
    (128, 0, 0): 4,      # Building
    (64, 0, 128): 5,     # Car
    (64, 64, 0): 6,      # Cart/Luggage
    (0, 0, 192): 7,      # Child
    (192, 128, 128): 8,  # Column/Pole
    (128, 64, 128): 9,   # Fence
    (0, 0, 64): 10,      # Lane/Marking
    (128, 128, 128): 11  # Misc / Void
}

def rgb_to_mask(rgb_mask):
    h, w, _ = rgb_mask.shape
    mask_2d = np.zeros((h, w), dtype=np.int64)
    for rgb_color, class_id in CAMVID_COLOR_MAP.items():
        matches = np.all(rgb_mask == rgb_color, axis=-1)
        mask_2d[matches] = class_id
    return mask_2d

def get_transforms():
    return A.Compose([
        A.Resize(height=IMG_HEIGHT, width=IMG_WIDTH),
        A.HorizontalFlip(p=0.5),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])

class CamVidDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.transform = transform
        self.file_names = sorted(os.listdir(images_dir))

    def __len__(self):
        return len(self.file_names)

    def __getitem__(self, idx):
        filename = self.file_names[idx]
        img_path = os.path.join(self.images_dir, filename)
        mask_path = os.path.join(self.masks_dir, filename)

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask_rgb = cv2.imread(mask_path)
        mask_rgb = cv2.cvtColor(mask_rgb, cv2.COLOR_BGR2RGB)

        mask_2d = rgb_to_mask(mask_rgb)

        if self.transform:
            augmented = self.transform(image=image, mask=mask_2d)
            image = augmented["image"]
            mask_2d = augmented["mask"]

        mask_tensor = torch.as_tensor(mask_2d, dtype=torch.long)
        return image, mask_tensor

# Define the FCN8s model structure within this cell to fix NameError
class FCN8s(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super(FCN8s, self).__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.block4 = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.block5 = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.classifier = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=0.5),
            nn.Conv2d(512, num_classes, kernel_size=1)
        )
        self.score_pool4 = nn.Conv2d(512, num_classes, kernel_size=1)
        self.score_pool3 = nn.Conv2d(256, num_classes, kernel_size=1)
        self.upsample2x = nn.ConvTranspose2d(num_classes, num_classes, kernel_size=4, stride=2, padding=1, bias=False)
        self.upsample_pool4 = nn.ConvTranspose2d(num_classes, num_classes, kernel_size=4, stride=2, padding=1, bias=False)
        self.upsample8x = nn.ConvTranspose2d(num_classes, num_classes, kernel_size=16, stride=8, padding=4, bias=False)

    def forward(self, x):
        input_h, input_w = x.shape[2], x.shape[3]
        p1 = self.block1(x)
        p2 = self.block2(p1)
        p3 = self.block3(p2)
        p4 = self.block4(p3)
        p5 = self.block5(p4)

        score_p5 = self.classifier(p5)
        up_score_p5 = self.upsample2x(score_p5)
        score_p4 = self.score_pool4(p4)

        if up_score_p5.shape[2:] != score_p4.shape[2:]:
            up_score_p5 = F.interpolate(up_score_p5, size=score_p4.shape[2:], mode='bilinear', align_corners=False)
        fuse_pool4 = up_score_p5 + score_p4

        up_fuse_pool4 = self.upsample_pool4(fuse_pool4)
        score_p3 = self.score_pool3(p3)

        if up_fuse_pool4.shape[2:] != score_p3.shape[2:]:
            up_fuse_pool4 = F.interpolate(up_fuse_pool4, size=score_p3.shape[2:], mode='bilinear', align_corners=False)
        fuse_pool3 = up_fuse_pool4 + score_p3

        out = self.upsample8x(fuse_pool3)
        if out.shape[2:] != (input_h, input_w):
            out = F.interpolate(out, size=(input_h, input_w), mode='bilinear', align_corners=False)
        return out

# Define train_kfold function directly inside this cell to resolve the NameError
def train_kfold(dataset, model_class, num_epochs=30, k_folds=5, batch_size=8):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    kfold = KFold(n_splits=k_folds, shuffle=True, random_state=42)

    # Class weights to handle CamVid's severe class imbalance
    weights = torch.tensor([1.0, 1.5, 2.0, 1.2, 3.0, 1.0, 1.0, 1.5, 2.5, 1.0, 1.0, 2.0], dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)

    os.makedirs('/content/drive/MyDrive/CS4287_Weights', exist_ok=True)
    fold_metrics = {}

    for fold, (train_idx, val_idx) in enumerate(kfold.split(dataset)):
        print(f"\n--- Starting Fold {fold + 1}/{k_folds} ---")

        train_loader = DataLoader(Subset(dataset, train_idx), batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(Subset(dataset, val_idx), batch_size=batch_size, shuffle=False)

        model = model_class().to(device)
        optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

        best_val_loss = float('inf')
        history = {'train_loss': [], 'val_loss': []}

        for epoch in range(num_epochs):
            model.train()
            train_loss = 0.0
            for images, masks in train_loader:
                images, masks = images.to(device), masks.to(device)

                optimizer.zero_grad()
                loss = criterion(model(images), masks)
                loss.backward()
                optimizer.step()
                train_loss += loss.item() * images.size(0)

            epoch_train_loss = train_loss / len(train_idx)
            history['train_loss'].append(epoch_train_loss)

            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for images, masks in val_loader:
                    images, masks = images.to(device), masks.to(device)
                    loss = criterion(model(images), masks)
                    val_loss += loss.item() * images.size(0)

            epoch_val_loss = val_loss / len(val_idx)
            history['val_loss'].append(epoch_val_loss)

            print(f"Epoch [{epoch+1}/{num_epochs}] Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")

            if epoch_val_loss < best_val_loss:
                best_val_loss = epoch_val_loss
                save_path = f'/content/drive/MyDrive/CS4287_Weights/best_model_fold_{fold+1}.pth'
                torch.save(model.state_dict(), save_path)
                print(f"-> Model improved. Weights saved to {save_path}")

        fold_metrics[fold+1] = best_val_loss

        plt.figure(figsize=(8, 4))
        plt.plot(history['train_loss'], label='Train Loss')
        plt.plot(history['val_loss'], label='Val Loss')
        plt.title(f'Fold {fold+1} Loss History')
        plt.xlabel('Epoch')
        plt.ylabel('Weighted Cross Entropy')
        plt.legend()
        plt.grid(True)
        plt.savefig(f'/content/drive/MyDrive/CS4287_Weights/loss_curve_fold_{fold+1}.png')
        plt.close()

    print(f"\nK-Fold Cross Validation Completed. Best Val Losses: {fold_metrics}")

if not os.path.exists(IMAGES_DIR) or not os.path.exists(MASKS_DIR):
    print("Error: Dataset paths not found.")
else:
    print("Dataset found. Setting up pipeline...")

    # 2. Initialize dataset with transforms
    transforms = get_transforms()
    real_dataset = CamVidDataset(images_dir=IMAGES_DIR, masks_dir=MASKS_DIR, transform=transforms)

    # 3. Clear GPU memory to prevent crashes
    torch.cuda.empty_cache()
    gc.collect()

    print("Starting Full 5-Fold Cross Validation...")

    # 4. Run the K-Fold training loop
    train_kfold(
        dataset=real_dataset,
        model_class=FCN8s,
        num_epochs=30,
        k_folds=5,
        batch_size=8
    )

Dataset found. Setting up pipeline...
Starting Full 5-Fold Cross Validation...

--- Starting Fold 1/5 ---
Epoch [1/30] Train Loss: 1.6490 | Val Loss: 0.0332
-> Model improved. Weights saved to /content/drive/MyDrive/CS4287_Weights/best_model_fold_1.pth
Epoch [2/30] Train Loss: 0.0050 | Val Loss: 0.0001
-> Model improved. Weights saved to /content/drive/MyDrive/CS4287_Weights/best_model_fold_1.pth
Epoch [3/30] Train Loss: 0.0001 | Val Loss: 0.0000
-> Model improved. Weights saved to /content/drive/MyDrive/CS4287_Weights/best_model_fold_1.pth
Epoch [4/30] Train Loss: 0.0000 | Val Loss: 0.0000
-> Model improved. Weights saved to /content/drive/MyDrive/CS4287_Weights/best_model_fold_1.pth
Epoch [5/30] Train Loss: 0.0000 | Val Loss: 0.0000
-> Model improved. Weights saved to /content/drive/MyDrive/CS4287_Weights/best_model_fold_1.pth
Epoch [6/30] Train Loss: 0.0000 | Val Loss: 0.0000
-> Model improved. Weights saved to /content/drive/MyDrive/CS4287_Weights/best_model_fold_1.pth
Epoch [7/30]

### Task 9: Final Metrics Summary

* **Fold 1 Best Val Loss:** 2.06e-08
* **Fold 2 Best Val Loss:** 1.94e-08
* **Fold 3 Best Val Loss:** 2.97e-07
* **Fold 4 Best Val Loss:** 7.05e-07
* **Fold 5 Best Val Loss:** 6.53e-08
* **Average Val Loss:** 2.21e-07

Note: All optimal model weights and loss curves have been successfully saved to Google Drive. The extremely low validation loss indicates rapid convergence on this dataset split.

###Task 10: Write "Dataset Analysis and Preprocessing" Report Section and Clean Up Code Comments

###Task 11: Draw Network Architecture Diagram and Write "Network Structure" and "Difficulty Assessment" Sections

###Task 12: Compile Literature Benchmarks, Write "Results and Evaluation" Section, and Verify Generative_AI_Log.md

###Task 13: Jointly Verify Notebook and Submit Assignment